# Generación de Espectros de Transmisión con TauREx 3
En este notebook calcularemos y compararemos los espectros de transmisión de TRAPPIST-1e para distintos escenarios de agricultura utilizando TauREx 3.

In [1]:
import taurex
from taurex.cache import OpacityCache, CIACache
from taurex.model import TransmissionModel
from taurex.data.stellar import Star
from taurex.data.planet import Planet
from taurex.pressure import ArrayPressureProfile
from taurex.temperature import TemperatureArray
from taurex.chemistry import TaurexChemistry, ArrayGas
from taurex.contributions import AbsorptionContribution, CIAContribution, RayleighContribution

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configurar rutas de opacidades (xsec y CIA)
# Actualiza estas rutas si es necesario
xsec_path = "D:\\Proyectos\\IA_SpecAtm_Bio\\Data\\Opacities Taurex"
cia_path = "D:\\Proyectos\\IA_SpecAtm_Bio\\Data\\CIA"

OpacityCache().clear_cache()
OpacityCache().set_opacity_path(xsec_path)
CIACache().set_cia_path(cia_path)

ModuleNotFoundError: No module named 'taurex'

In [ ]:
#***** Definir propiedades de la estrella y el planeta *****#

# Constantes
R_Sun = 6.957e8 # m
R_E = 6.371e6 # m
M_E = 5.972e24 # kg
M_Jup = 1.898e27 # kg
R_Jup = 7.1492e7 # m

# Propiedades de la estrella (TRAPPIST-1)
R_s = 0.11697 # Radios solares
T_s = 2559.0  # K
M_s = 0.0898  # Masas solares (aprox para TRAPPIST-1)

star = Star(temperature=T_s, radius=R_s, mass=M_s)

# Propiedades del planeta (TRAPPIST-1e)
R_p = 0.917985 * R_E / R_Jup # Radios de Júpiter
M_p = 0.6356 * M_E / M_Jup   # Masas de Júpiter

planet = Planet(planet_mass=M_p, planet_radius=R_p)

In [ ]:
#***** Función para leer perfiles y crear el modelo TauREx *****#

def build_taurex_model(pt_file, chem_file, molecules):
    # Leer perfil P-T
    df_pt = pd.read_csv(pt_file, delim_whitespace=True)
    # Las columnas son ALT, P, T
    p_bar = df_pt['P'].values
    t_array = df_pt['T'].values
    
    # Convertir presión a Pascales (TauREx usa Pa)
    p_pa = p_bar * 1e5
    
    # Leer perfil de química
    df_chem = pd.read_csv(chem_file, delim_whitespace=True)
    
    # Crear perfil de presión exacto
    # TauREx espera que la presión esté ordenada de TOA a superficie o viceversa, 
    # ArrayPressureProfile lo maneja.
    pressure_profile = ArrayPressureProfile(array=p_pa)
    
    # Crear perfil de temperatura
    temperature_profile = TemperatureArray(tp_array=t_array, p_points=p_pa)
    
    # Crear química
    chemistry = TaurexChemistry(fill_gases=['N2'])
    
    for mol in molecules:
        if mol in df_chem.columns:
            mix_ratio = df_chem[mol].values
            gas = ArrayGas(molecule_name=mol, mix_ratio_array=mix_ratio)
            chemistry.addGas(gas)
        else:
            print(f"Advertencia: {mol} no encontrado en {chem_file}")
            
    # Ensamblar el modelo de transmisión
    model = TransmissionModel(
        planet=planet,
        star=star,
        pressure_profile=pressure_profile,
        temperature_profile=temperature_profile,
        chemistry=chemistry
    )
    
    # Añadir contribuciones (Absorción, CIA, Rayleigh)
    model.add_contribution(AbsorptionContribution())
    # model.add_contribution(CIAContribution(cia_pairs=['H2-H2', 'H2-He'])) # Descomentar y ajustar pares CIA según disponibilidad
    model.add_contribution(RayleighContribution())
    
    # Construir el modelo
    model.build()
    
    return model

In [ ]:
#***** Definir escenarios y construir modelos *****#

base_dir = '../profiles/'
scenarios = {
    'PreAgri': {'pt': 'Trappist_A0_PreAgri_PT.txt', 'chem': 'Trappist_A0_PreAgri_chem.txt', 'color': 'darkgreen'},
    'Current': {'pt': 'Trappist_A1_Current_PT.txt', 'chem': 'Trappist_A1_Current_chem.txt', 'color': 'orangered'},
    'Moderate': {'pt': 'Trappist_A2_Moderate_PT.txt', 'chem': 'Trappist_A2_Moderate_chem.txt', 'color': 'darkblue'},
    'Extreme': {'pt': 'Trappist_A3_Extreme_PT.txt', 'chem': 'Trappist_A3_Extreme_chem.txt', 'color': 'darkred'}
}

molecules_to_include = ['H2O', 'CO2', 'CH4', 'O2', 'O3', 'N2O', 'NH3']

models = {}
for name, files in scenarios.items():
    print(f"Construyendo modelo para: {name}")
    pt_path = base_dir + files['pt']
    chem_path = base_dir + files['chem']
    
    models[name] = build_taurex_model(pt_path, chem_path, molecules_to_include)

In [ ]:
#***** Calcular espectros *****#

spectra = {}
for name, model in models.items():
    print(f"Calculando espectro para: {name}")
    # model.model() devuelve (native_grid, depth, tau, extra)
    result = model.model()
    wngrid = result[0]
    transit_depth = result[1]
    
    # Convertir número de onda (cm^-1) a longitud de onda (um)
    wl_um = 10000.0 / wngrid
    
    # Ordenar por longitud de onda para el plot
    sort_idx = np.argsort(wl_um)
    
    spectra[name] = {
        'wl': wl_um[sort_idx],
        'depth': transit_depth[sort_idx]
    }

In [ ]:
#***** Graficar espectros *****#

plt.figure(figsize=(12, 6))

for name, data in spectra.items():
    color = scenarios[name]['color']
    plt.plot(data['wl'], data['depth'], label=f'{name} Earth', color=color, linewidth=2, alpha=0.85)

plt.xscale('log')
plt.xlim(0.6, 14.0)
# plt.ylim(5.16e-3, 5.32e-3) # Ajustar límites Y según los resultados de TauREx

plt.xlabel('Wavelength ($\mu$m)', fontsize=14)
plt.ylabel('Transit Depth $(R_p/R_s)^2$', fontsize=14)
plt.title('Transmission Spectra (TauREx 3)', fontsize=16)
plt.legend(loc='upper right', fontsize=12)
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()